# 08 Custom Chattiness Evaluator

This notebook defines a custom evaluator for response chattiness, loads prior JSON output, and batch-scores both model responses per pair.

The output is displayed using the same renderer style as Notebook 05.

## Load Prior Output

In [34]:
import json
from pathlib import Path

pairs_output_path = Path('../outputs/similarity-pairs-results.json')
if not pairs_output_path.exists():
    raise RuntimeError('Missing ../outputs/similarity-pairs-results.json. Run 07_pairwise_similarity.ipynb first.')

rows = json.loads(pairs_output_path.read_text(encoding='utf-8'))
print(f'Loaded {len(rows)} rows from {pairs_output_path}')

required = {'pair', 'query', 'response_4o', 'response_51'}
missing = [i + 1 for i, r in enumerate(rows) if not required.issubset(r.keys())]
if missing:
    raise RuntimeError(f'Rows missing required fields at indices: {missing}')

rows[0].keys()

Loaded 5 rows from ..\outputs\similarity-pairs-results.json


dict_keys(['pair', 'query', 'similarity', 'evaluator_reason', 'response_4o', 'response_51'])

## Define Custom Evaluator

In [35]:
import re
from dataclasses import dataclass

@dataclass
class ChattinessResult:
    score: float
    words: int
    sentences: int
    avg_words_per_sentence: float
    filler_hits: int
    reason: str

class ChattinessEvaluator:
    """Simple deterministic evaluator where higher score means more verbose/chattier output."""

    filler_terms = [
        'basically', 'actually', 'just', 'really', 'very', 'in general',
        'for example', 'in practice', 'overall', 'typically'
    ]

    def evaluate(self, text: str) -> ChattinessResult:
        clean = (text or '').strip()
        words = len(re.findall(r"\b\w+[\w'-]*\b", clean))
        sentences = max(1, len([s for s in re.split(r"[.!?]+", clean) if s.strip()]))
        avg_wps = words / sentences if sentences else float(words)

        lower = clean.lower()
        filler_hits = sum(lower.count(term) for term in self.filler_terms)

        # Calibrated to avoid ceiling effects and preserve model-to-model spread.
        # Typical score band in this demo should be around 2.0 - 4.5 rather than saturating at 5.
        raw = 1.0 + (words / 55.0) + (avg_wps / 18.0) + (filler_hits * 0.2)
        score = round(min(5.0, max(1.0, raw)), 2)

        reason = (
            f'words={words}, sentences={sentences}, avg_words/sentence={avg_wps:.1f}, '
            f'filler_hits={filler_hits}, raw={raw:.2f}'
        )

        return ChattinessResult(
            score=score,
            words=words,
            sentences=sentences,
            avg_words_per_sentence=round(avg_wps, 2),
            filler_hits=filler_hits,
            reason=reason,
        )

## Batch Evaluate Both Models

In [36]:
evaluator = ChattinessEvaluator()

chattiness_rows = []
for row in rows:
    r4o = evaluator.evaluate(row['response_4o'])
    r51 = evaluator.evaluate(row['response_51'])

    chattier = 'gpt-4o' if r4o.score > r51.score else 'gpt-5.1' if r51.score > r4o.score else 'tie'

    chattiness_rows.append({
        'pair': row['pair'],
        'query': row['query'],
        'response_4o': row['response_4o'],
        'response_51': row['response_51'],
        'chattiness_4o': r4o.score,
        'chattiness_51': r51.score,
        'words_4o': r4o.words,
        'words_51': r51.words,
        'sentences_4o': r4o.sentences,
        'sentences_51': r51.sentences,
        'avg_wps_4o': r4o.avg_words_per_sentence,
        'avg_wps_51': r51.avg_words_per_sentence,
        'filler_hits_4o': r4o.filler_hits,
        'filler_hits_51': r51.filler_hits,
        'reason_4o': r4o.reason,
        'reason_51': r51.reason,
        'chattier_model': chattier,
    })

print(f'Batch-evaluated {len(chattiness_rows)} rows.')
for r in chattiness_rows:
    print(
        f"Pair {r['pair']}: "
        f"4o={r['chattiness_4o']} (w={r['words_4o']}, s={r['sentences_4o']}) vs "
        f"5.1={r['chattiness_51']} (w={r['words_51']}, s={r['sentences_51']}) -> {r['chattier_model']}"
    )

Batch-evaluated 5 rows.
Pair 1: 4o=2.95 (w=53, s=3) vs 5.1=3.51 (w=63, s=3) -> gpt-5.1
Pair 2: 4o=2.69 (w=46, s=3) vs 5.1=3.54 (w=80, s=5) -> gpt-5.1
Pair 3: 4o=4.03 (w=124, s=12) vs 5.1=5.0 (w=230, s=15) -> gpt-5.1
Pair 4: 4o=2.54 (w=42, s=3) vs 5.1=3.28 (w=62, s=3) -> gpt-5.1
Pair 5: 4o=2.2 (w=41, s=5) vs 5.1=3.32 (w=70, s=6) -> gpt-5.1


## Display In Notebook 05 Style

In [37]:
import importlib
import sys
from IPython.display import Markdown, display

app_dir = Path('../app').resolve()
if str(app_dir) not in sys.path:
    sys.path.append(str(app_dir))

import comparison_renderer
importlib.reload(comparison_renderer)

avg_4o = sum(r['chattiness_4o'] for r in chattiness_rows) / len(chattiness_rows)
avg_51 = sum(r['chattiness_51'] for r in chattiness_rows) / len(chattiness_rows)
avg_delta = avg_51 - avg_4o

if avg_delta > 0:
    overall = 'gpt-5.1 is chattier on average.'
elif avg_delta < 0:
    overall = 'gpt-4o is chattier on average.'
else:
    overall = 'Both are equally chatty on average.'

# Show the key result in a prominent summary block before the detailed cards.
display(Markdown(
    '## Key Result\n'
    f'**{overall}**\n\n'
    f'- Average chattiness score (gpt-4o): **{avg_4o:.2f}**\n'
    f'- Average chattiness score (gpt-5.1): **{avg_51:.2f}**\n'
    f'- Average delta (5.1 - 4o): **{avg_delta:+.2f}**'
))

render_rows = []
for r in chattiness_rows:
    diff = r['chattiness_51'] - r['chattiness_4o']
    winner = 'gpt-5.1' if diff > 0 else 'gpt-4o' if diff < 0 else 'tie'

    word_delta = r['words_51'] - r['words_4o']
    sent_delta = r['sentences_51'] - r['sentences_4o']
    avg_wps_delta = r['avg_wps_51'] - r['avg_wps_4o']
    filler_delta = r['filler_hits_51'] - r['filler_hits_4o']

    if winner == 'tie':
        headline = 'Result: both models are equally chatty on this prompt.'
    else:
        headline = f"Result: {winner} is chattier by {abs(diff):.2f} points on this prompt."

    eval_note = (
        f"{headline}\n"
        f"Score comparison: gpt-4o={r['chattiness_4o']} vs gpt-5.1={r['chattiness_51']} (delta={diff:+.2f})\n"
        f"Why: word delta={word_delta:+d}, sentence delta={sent_delta:+d}, "
        f"avg words/sentence delta={avg_wps_delta:+.2f}, filler delta={filler_delta:+d}.\n"
        f"gpt-4o -> words={r['words_4o']}, sentences={r['sentences_4o']}, "
        f"avg words/sentence={r['avg_wps_4o']}, filler hits={r['filler_hits_4o']}\n"
        f"gpt-5.1 -> words={r['words_51']}, sentences={r['sentences_51']}, "
        f"avg words/sentence={r['avg_wps_51']}, filler hits={r['filler_hits_51']}"
    )

    render_rows.append({
        'scenario_id': f"pair_{r['pair']:02d}",
        'category': 'Custom Evaluator: Chattiness',
        'system_label': 'Custom evaluator',
        'system': 'ChattinessEvaluator (higher = more verbose)',
        'user_label': 'Prompt',
        'user': r['query'],
        'difference_label': 'Custom evaluation summary',
        'difference_explanation': eval_note,
        'badges': [],
        'primary_deployment': 'chat4o',
        'secondary_deployment': 'chat51',
        'primary_status': 'success',
        'secondary_status': 'success',
        'primary_response': r['response_4o'],
        'secondary_response': r['response_51'],
    })

comparison_renderer.render_comparison(render_rows)

## Key Result
**gpt-5.1 is chattier on average.**

- Average chattiness score (gpt-4o): **2.88**
- Average chattiness score (gpt-5.1): **3.73**
- Average delta (5.1 - 4o): **+0.85**

gpt-4o - chat4o,gpt-5.1 - chat51
"Put retry logic in your service code. You have full control over which status codes and exception types are worth retrying, you can log intermediate attempts, and you avoid the operational overhead of running a service mesh for a single service. A short try/sleep loop or the tenacity library covers most cases.","Delegate retries to the service mesh (Istio or Linkerd) rather than duplicating the logic in every service. Retries are an infrastructure concern: centralising them in the mesh means consistent behaviour across all services, configurable via YAML without a redeploy, and built-in observability and circuit breaking at no extra code cost. If you are already on Kubernetes, this is the right layer for it."
gpt-4o - chat4o,gpt-5.1 - chat51
"For scraping many URLs, async is significantly faster. I/O wait dominates in scraping, so asyncio with aiohttp or httpx lets you fire hundreds of requests concurrently without spawning hundreds of threads. Use sync requests only for a handful of URLs or a quick one-off script.","It depends on scale. For fewer than roughly 20 URLs, sync requests with a ThreadPoolExecutor is simpler and fast enough. For hundreds of URLs, asyncio with aiohttp or httpx offers better throughput. For large crawls, Scrapy manages concurrency, politeness delays, and pipelines for you so you don't have to wire up the event loop manually. Async raw code is not always the right answer — the added complexity is only worth it when you have the URL volume to justify it."
gpt-4o - chat4o,gpt-5.1 - chat51
"Mia stood at the back of Room 12, palms damp around her note cards. Every laugh from the hallway sounded like a warning. She had practiced at home, but now the words felt slippery. When her name was called, she walked to the front and saw her friend Theo in the first row, giving her a small thumbs-up. Mia took one breath, then another. She opened with the story of her grandmother teaching her to grow tomatoes on a tiny balcony. Something shifted. The class leaned in. Her voice steadied. By the last slide, she was speaking without looking down. When she finished, there was a beat of silence and then applause. Mia sat down smiling, surprised to realize she wanted to present again.","The projector hummed like a nervous insect while Leila waited for her turn. Her presentation on urban gardens was next, and she had already imagined three disasters: forgetting every line, dropping her flash drive, and fainting in front of twenty-seven classmates. She pressed her fingers to the edge of her notebook and remembered what her older brother had told her that morning: 'You do not need to be fearless. You just need your first sentence.' When the teacher called her name, Leila walked to the front, knees unsteady, and clicked to the title slide. 'Last summer, a tomato plant grew through a crack in our apartment parking lot,' she began. A few students looked up. She continued, describing how something fragile could thrive in hard places. By slide three she stopped reading her notes. By slide five she was answering questions before they were fully asked. She compared rooftop gardens to open-air classrooms, talked about food deserts, and explained why planting herbs outside the library had changed how younger students used the space. At the end, the room was quiet in the good way - attentive, not awkward. Then came claps, then more claps, and even the students who usually stared at their phones were looking at her. Back at her desk, Leila wrote a single line in the margin of her notebook: 'Confidence is what shows up after you start.'"
gpt-4o - chat4o,gpt-5.1 - chat51
"Yes, for most APIs Python is fast enough. Async frameworks like FastAPI handle high concurrency well, and the bottleneck is usually the database or network, not the Python runtime. Profile before assuming Python is too slow — premature optimisation leads to unnecessary complexity.","Python has real performance limits for high-throughput sc

## Save Custom Evaluator Output

In [30]:
custom_output_path = Path('../outputs/custom-chattiness-results.json')
custom_output_path.write_text(json.dumps(chattiness_rows, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Saved custom evaluator results to {custom_output_path}')

Saved custom evaluator results to ..\outputs\custom-chattiness-results.json
